# Лучшее решение: View Codes GRU + LightGBM

Public Accuracy: **0.46172**. Private Accuracy: **0.40311**.


In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

TASK_DIR = Path.cwd()
if not (TASK_DIR / "solution/models/manifest.json").exists():
    TASK_DIR = TASK_DIR / "task 3"
ROOT = TASK_DIR / "solution"

assert (ROOT / "models/manifest.json").exists(), (
    "Открой ноутбук из проекта raif или папки task 3"
)

runtime = Path("/mnt/d/raif-gru/site-packages")
if runtime.exists():
    sys.path.insert(0, str(runtime))

sys.path.insert(0, str(ROOT))

manifest = json.loads((ROOT / "models/manifest.json").read_text())

print(manifest["recipe"])
print("Public:", manifest["public_score"])
print("Кандидатов:", len(manifest["candidate_products"]))


75% View Codes GRU +25% LightGBM
Public: 0.46172
Кандидатов: 42


## Как устроен прогноз

GRU с вниманием читает последние **160 событий** клиента. Для каждого продукта она получает **114 признаков** и общий контекст истории. LightGBM использует частоты, давность, сессии и последние действия.

Оцениваем все **42 продукта**, смешиваем результаты: `0.75 × GRU + 0.25 × LightGBM`, затем выбираем максимальный. При адаптации использовались известные прошлые целевые клики из входных историй test.


## Обучение моего решения: GRU + View Codes + LightGBM

Код признаков и моделей остаётся в `.py`, а здесь видны порядок действий, параметры и вызовы обучения.

1. Строим агрегаты, последовательности из последних 160 событий и исторические примеры.
2. Обучаем GRU с вниманием и 110 признаками: 21 эпоха, `lr=0.001`.
3. Добавляем четыре View Codes признака и дообучаем 5 эпох с `lr=0.0001`.
4. Адаптируем GRU на известных поздних кликах из историй test: 24 эпохи, `lr=0.0003`.
5. Обучаем LightGBM на train и добавляем 27 деревьев на тех же поздних кликах.
6. Смешиваем вероятности: `0.75 × GRU + 0.25 × LightGBM`.

Запуск всех ячеек ниже запускает обучение. Нужны `train.parquet`,
`test.parquet`, `event_dictionary.parquet`, `sample_submission.csv` в `solution/data/` и зависимости
из `solution/requirements_gru.txt`. Построение кэшей и обучение могут занять значительное время;
GRU использует CUDA, если она доступна. Новые кэши, веса, журналы и CSV записываются
в `task 3/solution/artifacts/notebook_training/`, отдельно от сохранённого лучшего решения.

In [2]:
import gc
import os
import numpy as np

TRAIN_DIR = ROOT / "artifacts" / "notebook_training"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
os.environ["RAIF_GRU_ARTIFACTS"] = str(TRAIN_DIR)

import next_click_gru as gru

assert gru.OUT.resolve() == TRAIN_DIR.resolve(), "Перезапусти ядро и выполни ячейки по порядку"

# Старые функции LightGBM читают общий CACHE; направляем его в каталог нового запуска.
gru.CACHE = TRAIN_DIR / "lightgbm"
gru.CACHE.mkdir(exist_ok=True)

import next_click_model as tabular
from next_click_extended import build_extended
from next_click_behavior import build_cache as build_behavior_cache, load_behavior
from next_click_view_codes import build_cache as build_view_cache, load_codes, expanded_base
from next_click_gru_finish import legacy_table, fit_legacy, predict_legacy

tabular.ARTIFACTS = gru.CACHE
print("Устройство:", gru.DEVICE)
print("Артефакты обучения:", TRAIN_DIR)


Устройство: cuda
Артефакты обучения: /home/nikita/proj/raif/task 3/solution/artifacts/notebook_training


### Подготовка признаков и исторических примеров

Для GRU используем 41 агрегат, 41 индикатор пропуска, 28 поведенческих признаков
и затем четыре View Codes признака. LightGBM получает собственную таблицу пар
«клиент, продукт» с категориальным контекстом.

Дополнительный обучающий пример, это префикс перед уже известным целевым кликом 469.
Пять событий непосредственно перед этим кликом исключаем.

In [3]:
# Исходный построитель таблицы LightGBM читает data/ относительно текущей папки.
previous_directory = Path.cwd()
try:
    os.chdir(ROOT)
    tabular.build_feature_cache()
finally:
    os.chdir(previous_directory)

gru.build_sequences()
build_extended()
build_behavior_cache()
build_view_cache()


BEHAVIOR_FEATURES_SELF_CHECK_OK


### Основная GRU

Объединяем официальные примеры train с префиксами его исторических кликов.
Обучаем GRU с вниманием к продукту-кандидату, hidden=64, seed=42.
Число эпох фиксировано по выбранному рецепту.


In [ ]:
full = load_behavior("train")
history = load_behavior("train", history=True)
source_train = gru.concatenate(full, history)
monitor = gru.take(full, np.arange(min(1024, len(full["y"]))))

assert source_train["x"].shape[1:] == (42, 110)
behavior_model, behavior_epoch = gru.fit(
    source_train, monitor, "notebook_behavior_source",
    epochs=21, lr=0.001, seed=42, fixed=True, attention=True, hidden=64,
    monitor_role="training_monitor",
)
assert behavior_epoch == 21
del full, history, source_train, monitor, behavior_model
gc.collect()


notebook_behavior_source {'epoch': 1, 'loss': 1.965287717991549, 'accuracy': 0.4580078125, 'seconds': 5.18, 'metric_role': 'training_monitor'}
notebook_behavior_source {'epoch': 5, 'loss': 1.6772060027779145, 'accuracy': 0.5146484375, 'seconds': 4.33, 'metric_role': 'training_monitor'}
notebook_behavior_source {'epoch': 10, 'loss': 1.62100670466136, 'accuracy': 0.52734375, 'seconds': 4.35, 'metric_role': 'training_monitor'}
notebook_behavior_source {'epoch': 15, 'loss': 1.5925684085672605, 'accuracy': 0.541015625, 'seconds': 4.38, 'metric_role': 'training_monitor'}


### Четыре View Codes признака: ещё 5 эпох

Добавляем число просмотров 399, 666 и 162, а также давность последнего просмотра 399
из видимого хвоста истории. `expanded_base` переносит обученные веса и добавляет
четыре входа с нулевыми весами, сохраняя исходный прогноз перед дообучением.
После этого обучаем модель с 114 признаками ещё пять эпох.


In [ ]:
full = load_codes("train")
history = load_codes("train", history=True)
source_train = gru.concatenate(full, history)
del full, history
gc.collect()

base = expanded_base("notebook_behavior_source")
monitor = gru.take(source_train, np.arange(min(128, len(source_train["y"]))))
source_model, source_epoch = gru.fit(
    source_train, monitor, "notebook_view_codes_source", base=base,
    epochs=5, lr=0.0001, seed=42, fixed=True, monitor_role="training_monitor",
)
assert source_epoch == 5 and len(source_model.x_mean) == 114
del source_train, monitor, base
gc.collect()


### Адаптация GRU на поздних известных кликах

Из историй test берём префиксы с известным целевым кликом начиная с 7 апреля 2026 года
(в исходных данных таких 746 примеров). Это прошлые клики во входных историях,
а не неизвестные ответы соревнования. Продолжаем обучение на 24 эпохи.

Здесь обучаем финальную модель на всех доступных поздних примерах. Для оценки качества
в экспериментах использовались отдельные клиентские и временные фолды.


In [ ]:
history = load_codes("test", history=True)
recent_indices = np.flatnonzero(history["meta"].click_time.to_numpy() >= gru.RECENT)
recent = gru.take(history, recent_indices)
del history
assert len(recent["y"]) > 0
assert (recent["meta"].end <= recent["meta"].click_time).all()
print("Примеров адаптации:", len(recent["y"]))

monitor = gru.take(recent, np.arange(min(128, len(recent["y"]))))
adapted_model, adapted_epoch = gru.fit(
    recent, monitor, "notebook_view_codes_adapted", base=source_model,
    epochs=24, lr=0.0003, seed=42, fixed=True, monitor_role="training_monitor",
)
assert adapted_epoch == 24
del source_model, monitor
gc.collect()


### LightGBM (473 исходных дерева + 27 адаптационных)

На официальном train строим строки «клиент, продукт»: правильный кандидат и десять
случайных отрицательных. Затем дообучаем дерево на тех же поздних примерах, что и GRU.


In [ ]:
context, matrix, meta, original_products = tabular.load_features("train")
source_booster = tabular.train_boosted(context, matrix, meta, original_products)
del context, matrix, meta
gc.collect()

history_table, products = legacy_table(history=True)
assert np.array_equal(products, recent["products"])
with np.load(gru.OUT / "test_extended.npz") as cached:
    expected_meta = cached["history_m"][recent_indices]
assert np.array_equal(expected_meta, recent["meta"].to_numpy())

adapted_booster = fit_legacy(
    source_booster, history_table[recent_indices], recent["y"], len(products)
)
adapted_booster.save_model(str(TRAIN_DIR / "notebook_lightgbm.txt"))
del source_booster, history_table, recent
gc.collect()


### Прогноз заново обученного ансамбля

Для каждого клиента оцениваем все 42 продукта. GRU возвращает softmax, бинарные
вероятности LightGBM нормируем и смешиваем с весами 0.75/0.25.
Выбираем продукт с максимальной итоговой вероятностью и выравниваем строки по `example_id`.


In [ ]:
test = load_codes("test")
neural_probabilities = gru.predict(adapted_model, test)
test_table, tree_products = legacy_table(history=False)
assert np.array_equal(tree_products, test["products"])
tree_probabilities = predict_legacy(adapted_booster, test_table)
probabilities = 0.75 * neural_probabilities + 0.25 * tree_probabilities

assert probabilities.shape == (len(test["y"]), len(tree_products))
assert np.isfinite(probabilities).all()
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
gru.write_submission(probabilities, tree_products, "submission_trained.csv")
trained_submission = pd.read_csv(TRAIN_DIR / "submission_trained.csv")
trained_submission.head()


## Сохранённое лучшее решение

Следующие ячейки показывают сохранённый CSV и воспроизводят его из готовых весов
в `solution/models/`. Они независимы от результатов нового обучения выше.


In [ ]:
best_submission = pd.read_csv(TASK_DIR / "submission_view_codes.csv")

assert len(best_submission) == 12359
assert best_submission.example_id.is_unique

print("Сохранённый файл:", TASK_DIR / "submission_view_codes.csv")
print("Строк:", len(best_submission))

best_submission.head()


Сохранённый файл: /home/nikita/proj/raif/task 3/submission_view_codes.csv
Строк: 12359


,example_id,encoded_product
0,0000559d3ae5245e50a54e3211c2e216,6
1,0000ffa325b571d1b0741d563ad8c354,3
2,00037570f0006ae92f8d69c4fc05ac72,63
3,0005cabcc6bba97d323d6fbb84a428b8,13
4,0007ce0e174df7aaf992dac3a92b5bd2,6


## Воспроизведение

Следующая ячейка читает исходные parquet, загружает веса из `solution/models/` и записывает отдельный CSV. Обучение не запускается. Программа сверяет результат с сохранённым лучшим файлом.

Из корня проекта в терминале: `bash "task 3/run_gru.sh"`.


In [ ]:
import predict_best

predict_best.run(ROOT / "submission_reproduced.csv")


BEST_REPRODUCED: 12359 rows, device=cuda, SHA256=9ef85077f8804769917cb4949953015e494387b3fe048a7e737847819889ae7f
/home/nikita/proj/raif/task 3/submission_reproduced.csv
